#**2nd Week**

##**Задачи - CTE (Продвинутый уровень)**

В этом тесте вам предстоит решить практические задачи продвинутого уровня на тему "CTE".

Найдите номера (room_number), которые были забронированы более 450 раз. Выведете всю информацию о данных номерах. Отсортировать по возрастанию номера комнаты.

In [ ]:
SELECT 
    r.*
FROM 
    rooms r
JOIN 
    (SELECT room_number
     FROM bookings
     GROUP BY room_number
     HAVING COUNT(*) > 450) b
ON 
    r.room_number = b.room_number
ORDER BY 
    r.room_number ASC;


Определить номера, которые были забронированы более шести раз в декабре 2023 года (учитывать по дате заезда). Вывести номера комнат и количество бронирований (booking_count)

In [ ]:
SELECT 
    b.room_number, 
    COUNT(*) AS booking_count
FROM 
    bookings b
WHERE 
    strftime('%Y-%m', b.check_in_date) = '2023-12'
GROUP BY 
    b.room_number
HAVING 
    COUNT(*) > 6
ORDER BY 
    b.room_number ASC;


Определить, в какие дни недели чаще всего бронируют номера. Вывести день недели (day_of_week) и количество бронирований (booking_count). Пронумеровать (rank()) получившуюся выборку по убыванию количества бронирований. Отсортировать в порядке возрастания ранга

In [ ]:
WITH booking_days AS (
    SELECT 
        strftime('%w', b.check_in_date) AS day_of_week,  -- Получаем день недели (0 - воскресенье, 6 - суббота)
        COUNT(*) AS booking_count
    FROM 
        bookings b
    GROUP BY 
        day_of_week
)
SELECT 
    CASE 
        WHEN day_of_week = '0' THEN '0'
        WHEN day_of_week = '1' THEN '1'
        WHEN day_of_week = '2' THEN '2'
        WHEN day_of_week = '3' THEN '3'
        WHEN day_of_week = '4' THEN '4'
        WHEN day_of_week = '5' THEN '5'
        WHEN day_of_week = '6' THEN '6'
    END AS day_of_week,
    booking_count,
    RANK() OVER (ORDER BY booking_count DESC) AS rank_num
FROM 
    booking_days
ORDER BY 
    rank_num ASC;


Определить отношение количества оценок со значением оценки 1 к общему количеству оценок в %. Итоговое значение отобразить в колонке ratio_ones_to_all. Округлить до двух знаков после запятой.

In [ ]:
SELECT 
    ROUND(
        (COUNT(CASE WHEN rating_value = 1 THEN 1 END) * 100.0) / COUNT(rating_value), 
        2
    ) AS ratio_ones_to_all
FROM ratings;


Определить процент клиентов, которые вернулись в отель (имеют более одной брони). Итоговое значение отобразить в колонке returning_client_percentage и округлить до двух знаков после запятой. Нормировать нужно на количество клиентов, которые делали бронь в нашем отеле.

In [ ]:
WITH client_bookings AS (
    SELECT 
        b.renter_id,
        COUNT(b.booking_id) AS booking_count
    FROM 
        bookings b
    GROUP BY 
        b.renter_id
)
SELECT 
    ROUND(
        (CAST(SUM(CASE WHEN cb.booking_count > 1 THEN 1 ELSE 0 END) AS FLOAT) / 
        COUNT(cb.renter_id)) * 100, 2
    ) AS returning_client_percentage
FROM 
    client_bookings cb;


Найти среднюю продолжительность проживания в днях для каждого месяца в году. Вывести месяц и среднюю продолжительность проживания в днях в колонке average_duration. Месяц сопоставляется с датами бронирования по дате заезда.

In [ ]:
SELECT 
    strftime('%m', b.check_in_date) AS month,
    AVG(julianday(b.check_out_date) - julianday(b.check_in_date)) AS average_duration
FROM 
    bookings b
GROUP BY 
    month
ORDER BY 
    month;
